In [ ]:
# Install packages with compatible versions for Colab
!pip install --upgrade pip
!pip install datasets transformers sacrebleu sentencepiece
# Install PyTorch version compatible with Colab's CUDA
!pip install --force-reinstall torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import os
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.nn.utils.rnn import pad_sequence

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define constants
BATCH_SIZE = 32
MAX_SEQ_LEN = 128  # Reasonable length for translation
EMBEDDING_SIZE = 512
NUM_HEADS = 8
FFN_HID_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
NUM_EPOCHS = 10
LEARNING_RATE = 0.0001
SUBSET_SIZE = 100000  # Use a subset of the data to fit on 3070

# Create directories for saving data and models
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Download and preprocess WMT14 English-French dataset using Hugging Face datasets
print("Loading WMT14 English-French dataset...")
dataset = load_dataset("wmt14", "fr-en")
print("Dataset loaded successfully!")

# Print dataset info
print(f"Dataset structure: {dataset}")

# Create tokenizers using Hugging Face transformers
tokenizer_en = AutoTokenizer.from_pretrained("t5-small")  # Using t5 tokenizer for English
tokenizer_fr = AutoTokenizer.from_pretrained("t5-small")  # Using t5 tokenizer for French

# Function to tokenize the datasets
def tokenize_dataset(examples):
    en_texts = []
    fr_texts = []
    
    # Extract texts from the correct structure
    for item in examples["translation"]:
        en_texts.append(item["en"])
        fr_texts.append(item["fr"])
    
    en_tokenized = tokenizer_en(en_texts, truncation=True, max_length=MAX_SEQ_LEN-2)
    fr_tokenized = tokenizer_fr(fr_texts, truncation=True, max_length=MAX_SEQ_LEN-2)
    
    return {
        "en_tokens": en_tokenized["input_ids"],
        "fr_tokens": fr_tokenized["input_ids"]
    }

# Tokenize train, validation and test datasets
tokenized_datasets = dataset.map(tokenize_dataset, batched=True)

# Create a function to filter by length and limit dataset size
def prepare_dataset(dataset_split, size):
    data = tokenized_datasets[dataset_split].select(range(min(size, len(tokenized_datasets[dataset_split]))))
    print(f"{dataset_split} samples: {len(data)}")
    
    src_data = []
    tgt_data = []
    
    for item in data:
        en_tokens = item["en_tokens"]
        fr_tokens = item["fr_tokens"]
        
        # Skip samples that are too long
        if len(en_tokens) <= MAX_SEQ_LEN and len(fr_tokens) <= MAX_SEQ_LEN:
            src_data.append(torch.tensor(en_tokens, dtype=torch.long))
            tgt_data.append(torch.tensor(fr_tokens, dtype=torch.long))
    
    return src_data, tgt_data

# Prepare the datasets
train_src, train_tgt = prepare_dataset("train", SUBSET_SIZE)
val_src, val_tgt = prepare_dataset("validation", SUBSET_SIZE // 10)
test_src, test_tgt = prepare_dataset("test", SUBSET_SIZE // 20)

# Get vocabulary sizes from tokenizers
src_vocab_size = tokenizer_en.vocab_size
tgt_vocab_size = tokenizer_fr.vocab_size
print(f"English vocabulary size: {src_vocab_size}")
print(f"French vocabulary size: {tgt_vocab_size}")

# Special token indices
PAD_IDX = tokenizer_en.pad_token_id
BOS_IDX = tokenizer_en.bos_token_id if tokenizer_en.bos_token_id else tokenizer_en.cls_token_id
EOS_IDX = tokenizer_en.eos_token_id if tokenizer_en.eos_token_id else tokenizer_en.sep_token_id

# Create batch function with padding
def collate_batch(batch):
    src_batch, tgt_batch = [], []
    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)
    
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

# Create DataLoaders
train_dataset = data.dataset.TensorDataset(train_src, train_tgt)
train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)

val_dataset = data.dataset.TensorDataset(val_src, val_tgt)
val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

test_dataset = data.dataset.TensorDataset(test_src, test_tgt)
test_loader = data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

# Transformer Model Definition
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, dim_feedforward, max_seq_length, dropout=0.1):
        super(TransformerModel, self).__init__()
        
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        self.out = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None,
                src_padding_mask=None, tgt_padding_mask=None,
                memory_key_padding_mask=None):
        
        src_emb = self.positional_encoding(self.src_embedding(src))
        tgt_emb = self.positional_encoding(self.tgt_embedding(tgt))
        
        output = self.transformer(src_emb, tgt_emb, src_mask, tgt_mask,
                                 None, src_padding_mask, tgt_padding_mask, memory_key_padding_mask)
        
        return self.out(output)
    
    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask
    
    def create_padding_mask(self, seq, pad_idx):
        return (seq == pad_idx)

# Initialize model
transformer = TransformerModel(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=EMBEDDING_SIZE,
    nhead=NUM_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=FFN_HID_DIM,
    max_seq_length=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(transformer.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)

# Helper function for creating masks
def create_masks(src, tgt):
    src_padding_mask = transformer.create_padding_mask(src, PAD_IDX)
    tgt_padding_mask = transformer.create_padding_mask(tgt, PAD_IDX)
    
    tgt_len = tgt.shape[1]
    tgt_mask = transformer.generate_square_subsequent_mask(tgt_len).to(device)
    
    return src_padding_mask, tgt_padding_mask, tgt_mask

# Evaluate initial loss before training
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in dataloader:
            src = src.to(device)
            tgt = tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            src_padding_mask, tgt_padding_mask, tgt_mask = create_masks(src, tgt_input)
            
            output = model(
                src, tgt_input, 
                tgt_mask=tgt_mask,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )
            
            loss = criterion(output.reshape(-1, tgt_vocab_size), tgt_output.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

# Evaluate initial loss
print("Evaluating initial loss before training...")
initial_val_loss = evaluate(transformer, val_loader)
print(f"Initial validation loss: {initial_val_loss:.4f}")

# Training loop
transformer.train()
best_val_loss = float('inf')

print("Starting training...")
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    transformer.train()
    
    for batch_idx, (src, tgt) in enumerate(train_loader):
        src = src.to(device)
        tgt = tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        # Create masks
        src_padding_mask, tgt_padding_mask, tgt_mask = create_masks(src, tgt_input)
        
        # Forward pass
        optimizer.zero_grad()
        output = transformer(
            src, tgt_input, 
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # Compute loss
        loss = criterion(output.reshape(-1, tgt_vocab_size), tgt_output.reshape(-1))
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Batch: {batch_idx}/{len(train_loader)}, " 
                  f"Loss: {loss.item():.4f}")
    
    # Validation
    val_loss = evaluate(transformer, val_loader)
    print(f"Epoch: {epoch+1}/{NUM_EPOCHS}, Training Loss: {epoch_loss/len(train_loader):.4f}, "
          f"Validation Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(transformer.state_dict(), 'models/transformer_best.pth')
        print(f"Saved new best model with validation loss: {val_loss:.4f}")

print("Training complete!")

# Load best model and evaluate on test set
transformer.load_state_dict(torch.load('models/transformer_best.pth'))
test_loss = evaluate(transformer, test_loader)
print(f"Test loss: {test_loss:.4f}")